<a href="https://colab.research.google.com/github/jenriver/tiny-transformer/blob/main/tiny_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import jax
import jax.numpy as jnp
import random

In [ ]:
jax.devices()

[CpuDevice(id=0)]

In [ ]:
chars_str = "0123456789+="
pad_str = "_"
MAX_LEN = len("999+999=1998")

def make_vocab(chars_str: str):
  return list(pad_str + chars_str)
vocab = make_vocab(chars_str)
print(vocab)
pad_id = vocab.index(pad_str)

['_', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '+', '=']


In [ ]:
def encode(s_in: str) -> list[int]:
  output = []
  for s in s_in:
    output.append(vocab.index(s))
  return output

def decode(ids: list[int]) -> str:
  s = ""
  for i in ids:
    s += vocab[i]
  return s

def make_example(max_digits=3) -> str:
  a = random.randint(0, 10**max_digits-1)
  b = random.randint(0, 10**max_digits-1)
  s = f'{a}+{b}={a+b}'
  # print(s)
  return s

def make_input_target(ids):
  return ids[:-1], ids[1:]

def pad_to(ids: list[int], T: int) -> tuple[list[int], list[int]]:
  n = len(ids)
  padded = ids + [pad_id] * (T-n)
  mask = [1] * n + [0] * (T-n)
  return padded, mask

def make_batch(B: int, T: int):
  inputs, targets, masks = [], [], []
  for _ in range(B):
    ids = encode(make_example())
    input, target = make_input_target(ids)
    input_padded, mask = pad_to(input, T)
    target_padded, _ = pad_to(target, T)
    inputs.append(input_padded)
    targets.append(target_padded)
    masks.append(mask)
  return jnp.array(inputs), jnp.array(targets), jnp.array(masks)

input = "12+34="
print(encode(input))
print(decode(encode(input)))

[2, 3, 11, 4, 5, 12]
12+34=


In [ ]:
def init_embed(key, V, D):
  return jax.random.normal(key, (V, D)) * (1.0 / jnp.sqrt(D))

def embed(table, ids):
  return table[ids]  # fancy-indexing gathers

def init_pos(key, T, D):
  return jax.random.normal(key, (T, D)) * (1.0 / jnp.sqrt(D))

def embed_tokens(tok_table, pos_table, ids):
  B, Tn = ids.shape
  x = tok_table[ids]
  x = x + pos_table[jnp.arange(Tn)]
  return x


In [ ]:
def init_attn(key, D, N, K, H):
  kq, kk, kv, ko = jax.random.split(key, 4)
  scale = 1.0 / jnp.sqrt(D)
  return {
      "Wq": jax.random.normal(kq, (D, N, H)) * scale,
      "Wk": jax.random.normal(kk, (D, K, H)) * scale,
      "Wv": jax.random.normal(kv, (D, K, H)) * scale,
      "Wo": jax.random.normal(ko, (N, H, D)) * scale,
  }

def attention(input: jnp.ndarray, attn: dict):
  B, T, D = input.shape
  q = jnp.einsum("btd,dnh->btnh", input, attn["Wq"])
  k = jnp.einsum("bsd,dkh->bskh", input, attn["Wk"])
  v = jnp.einsum("bsd,dnh->bsnh", input, attn["Wv"])

  attn_matrix = jnp.einsum("btnh,bsnh->btsn", q, k) / jnp.sqrt(H)
  S = k.shape[1]
  causal = jnp.tril(jnp.ones((T, S), dtype=bool))
  attn_matrix = jnp.where(causal[None, :, :, None], attn_matrix, -1e30)

  w = jax.nn.softmax(attn_matrix, axis=2)
  scaled_dot_product = jnp.einsum("btsn,bsnh->btnh", w, v)
  return jnp.einsum("btnh,nhd->btd", scaled_dot_product, attn["Wo"])


def init_mlp(key, D, F):
  kw1, kw2, kw3 = jax.random.split(key, 3)
  return {
      "w_gate": jax.random.normal(kw1, (D, F)) / jnp.sqrt(D),
      "w_up":   jax.random.normal(kw2, (D, F)) / jnp.sqrt(D),
      "w_down": jax.random.normal(kw3, (F, D)) / jnp.sqrt(F),
  }

def mlp(input: jnp.ndarray, p: dict):
  B, T, D = input.shape
  gate = jnp.einsum("btd,df->btf", input, p["w_gate"])
  up =   jnp.einsum("btd,df->btf", input, p["w_up"])
  return jnp.einsum("btf,fd->btd", jax.nn.silu(gate) * up, p["w_down"])

def init_norm(D):
  return {'g': jnp.ones(D)}

def rmsnorm(x, p, eps=1e-6):
  ms = jnp.mean(x**2, axis=-1, keepdims=True)
  return x * jax.lax.rsqrt(ms + eps) * p["g"]

def init_block(key, D, N, K, H, F):
  ka, km = jax.random.split(key, 2)
  return {
      "attn": init_attn(ka, D, N, K, H),
      "mlp":   init_mlp(km, D, F),
      "norm1":  init_norm(D),
      "norm2":  init_norm(D),
  }

def init_blocks(key, D, N, K, H, F, L):
  return [init_block(kb, D, N, K, H, F) for kb in jax.random.split(key, L)]

def block(x: jnp.ndarray, p: dict):
  x = x + attention(rmsnorm(x, p['norm1']), p['attn'])
  x = x + mlp(rmsnorm(x, p['norm2']), p['mlp'])
  return x

def init_model(key, V, T, D, N, K, H, F, L):
  kt, kp, kb, ku = jax.random.split(key, 4)
  return {
      "tok_table": init_embed(kt, V, D),
      "pos_table": init_embed(kp, T, D),
      "blocks": init_blocks(kb, D, N, K, H, F, L),
      "norm": init_norm(D),
      "unembed": jax.random.normal(ku, (D, V)) / jnp.sqrt(D),
  }

def model(ids, mp):
  x = embed_tokens(mp["tok_table"], mp["pos_table"], ids)
  for p in mp['blocks']:
    x = block(x, p)
  x = rmsnorm(x, mp['norm'])
  x = jnp.einsum('btd,dv->btv', x, mp['unembed'])
  return x


In [ ]:
B = 128
T = MAX_LEN - 1
D = 128
N = 4   # number of heads for query
H = 32  # 128 / 4 = 32 (head dim)
K = 4   # = N for MHA
F = 512 # 4 * D
L = 4
V = len(vocab)


ids, targets, masks = make_batch(B, T)
key = jax.random.PRNGKey(0)
params = init_model(key, V, T, D, N, K, H, F, L)
x = model(ids, params)
print(decode(jnp.argmax(x, axis=-1)[0].tolist()))

66666646464


In [ ]:
print(sum(p.size for p in jax.tree.leaves(params)))
print(x.shape)  # B, T, V


1054464
(128, 11, 13)


In [ ]:
print((V * D) + (T * D) + ((3 * D * F) + (4 * D * N * H) + (2 * D)) * L + D + (V * D))

1054464
